# Notebook 03 — Model Prediction
**Tujuan:** Bangun model prediksi churn (LR vs RF), evaluasi performa, hitung business impact.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import json
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, roc_curve, ConfusionMatrixDisplay)

OUT     = Path('../output')
FIGURES = OUT / 'figures'
BLUE, RED, LIGHT = '#2563EB', '#DC2626', '#93C5FD'

df = pd.read_parquet(OUT / 'df_clean.parquet')
print(f'Loaded: {len(df):,} baris')

## 1. Preprocessing

In [ ]:
# Kolom yang dipakai (drop ID dan kolom turunan yang redundan)
drop_cols = ['customerID', 'tenure_segment', 'risk_segment', 'Churn']
feature_cols = [c for c in df.columns if c not in drop_cols]

X = pd.get_dummies(df[feature_cols], drop_first=True)
y = df['Churn']

# Scale numerik
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges', 'num_services']
num_cols_encoded = [c for c in num_cols if c in X.columns]

scaler = StandardScaler()
X[num_cols_encoded] = scaler.fit_transform(X[num_cols_encoded])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {len(X_train):,} | Test: {len(X_test):,}')
print(f'Features: {X.shape[1]}')

## 2. Model 1 — Logistic Regression (Baseline)

In [ ]:
lr = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
lr.fit(X_train, y_train)

y_pred_lr  = lr.predict(X_test)
y_prob_lr  = lr.predict_proba(X_test)[:, 1]
auc_lr     = roc_auc_score(y_test, y_prob_lr)

print('=== Logistic Regression ===')
print(classification_report(y_test, y_pred_lr, target_names=['Retensi','Churn']))
print(f'ROC-AUC: {auc_lr:.4f}')

## 3. Model 2 — Random Forest

In [ ]:
rf = RandomForestClassifier(
    n_estimators=100, class_weight='balanced', random_state=42, n_jobs=-1
)
rf.fit(X_train, y_train)

y_pred_rf  = rf.predict(X_test)
y_prob_rf  = rf.predict_proba(X_test)[:, 1]
auc_rf     = roc_auc_score(y_test, y_prob_rf)

print('=== Random Forest ===')
print(classification_report(y_test, y_pred_rf, target_names=['Retensi','Churn']))
print(f'ROC-AUC: {auc_rf:.4f}')

## 4. Evaluasi Visual — Confusion Matrix & ROC Curve

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Confusion matrix LR
ConfusionMatrixDisplay(
    confusion_matrix(y_test, y_pred_lr),
    display_labels=['Retensi','Churn']
).plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title(f'Logistic Regression\nAUC: {auc_lr:.3f}', fontweight='bold')

# Confusion matrix RF
ConfusionMatrixDisplay(
    confusion_matrix(y_test, y_pred_rf),
    display_labels=['Retensi','Churn']
).plot(ax=axes[1], colorbar=False, cmap='Blues')
axes[1].set_title(f'Random Forest\nAUC: {auc_rf:.3f}', fontweight='bold')

# ROC Curve perbandingan
for model_name, y_prob, color in [
    ('Logistic Regression', y_prob_lr, LIGHT),
    ('Random Forest',       y_prob_rf, BLUE)
]:
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    axes[2].plot(fpr, tpr, linewidth=2, color=color, label=f'{model_name} (AUC={auc:.3f})')

axes[2].plot([0,1],[0,1], 'k--', linewidth=1, alpha=0.5)
axes[2].set_xlabel('False Positive Rate')
axes[2].set_ylabel('True Positive Rate')
axes[2].set_title('ROC Curve — LR vs RF', fontweight='bold')
axes[2].legend(loc='lower right')

plt.tight_layout()
plt.savefig(FIGURES / 'F_model_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: F_model_evaluation.png')

## 5. Feature Importance (Random Forest)

In [ ]:
feat_imp = pd.Series(rf.feature_importances_, index=X.columns)\
             .nlargest(10).sort_values()

fig, ax = plt.subplots(figsize=(10, 6))
colors_fi = [BLUE if i == len(feat_imp)-1 else LIGHT for i in range(len(feat_imp))]
ax.barh(feat_imp.index, feat_imp.values, color=colors_fi)
ax.set_xlabel('Feature Importance')
ax.set_title('Top 10 Faktor Prediksi Churn (Random Forest)', fontweight='bold')
for i, v in enumerate(feat_imp.values):
    ax.text(v + 0.001, i, f'{v:.3f}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig(FIGURES / 'G_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: G_feature_importance.png')

## 6. Business Impact Estimation

In [ ]:
# Prediksi pada seluruh dataset
X_all = pd.get_dummies(df[feature_cols], drop_first=True).reindex(columns=X.columns, fill_value=0)
X_all[num_cols_encoded] = scaler.transform(X_all[num_cols_encoded])

df['churn_prob'] = rf.predict_proba(X_all)[:, 1]
df['churn_pred'] = rf.predict(X_all)

at_risk     = df[df['churn_pred'] == 1]
revenue_risk = at_risk['MonthlyCharges'].sum()

print('=== Business Impact ===')
print(f'Pelanggan diprediksi churn : {len(at_risk):,}')
print(f'Revenue at risk (monthly)  : USD {revenue_risk:,.0f}')
print(f'Avg monthly charges        : USD {at_risk["MonthlyCharges"].mean():.2f}')

# Simpan metrics
from sklearn.metrics import precision_score, recall_score, f1_score
results = {
    'logistic_regression': {
        'auc':       round(auc_lr, 4),
        'precision': round(precision_score(y_test, y_pred_lr), 4),
        'recall':    round(recall_score(y_test, y_pred_lr), 4),
        'f1':        round(f1_score(y_test, y_pred_lr), 4)
    },
    'random_forest': {
        'auc':       round(auc_rf, 4),
        'precision': round(precision_score(y_test, y_pred_rf), 4),
        'recall':    round(recall_score(y_test, y_pred_rf), 4),
        'f1':        round(f1_score(y_test, y_pred_rf), 4)
    },
    'business_impact': {
        'at_risk_customers': int(len(at_risk)),
        'revenue_at_risk_monthly_usd': round(float(revenue_risk), 2)
    },
    'feature_importance_top10': feat_imp[::-1].to_dict()
}

with open(OUT / 'model_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print('\nSaved: model_results.json')